# IOS Risk Intelligence Core — Fine-Tuning Run
## Project 03: IOS Risk Brain #1 (Llama 3.1 8B Instruct + QLoRA)

This notebook trains **IOS Risk Brain #1** using **Unsloth + QLoRA** on Kaggle's free GPU tier (T4x2).

### Overview
- **Base Model:** `unsloth/Meta-Llama-3.1-8B-Instruct`
- **Dataset:** `Etherlabs/ios-risk-finetune-v3` (quality-gated multi-source pairs)
- **Technique:** QLoRA 4-bit Quantization + LoRA rank 16 adapters
- **Compute:** Kaggle T4 GPU (Zero Spend)

### Step 1: Install Dependencies
Install Unsloth, HuggingFace TRL, PEFT, Accelerate, and tracking tools.

In [ ]:
# Kaggle's image ships transformers 5.0.0 and datasets 5.0.0.
# Unsloth 2026.8.19 excludes transformers 5.0.0 by name (!=5.0.0) and caps
# datasets at <4.4.0, so a bare `pip install unsloth` has no resolvable
# solution and fails. Pin the whole compatible set in ONE command.
#
# xformers is pinned to 0.0.34, not "latest": 0.0.35 declares torch>=2.10 so
# pip happily picks it, but its compiled extensions are built for torch 2.11
# and it logs "Skipping import of cpp extensions due to incompatible torch
# version" on Kaggle's torch 2.10. 0.0.34 declares torch==2.10.0 exactly.
!pip install \
    "transformers==5.5.0" \
    "datasets==4.3.0" \
    "trl==0.24.0" \
    "bitsandbytes==0.50.1" \
    "xformers==0.0.34" \
    "peft>=0.18.0" \
    unsloth unsloth_zoo 2>&1 | tail -25

# Note: do NOT `--upgrade datasets` afterwards — that pulls 5.0.1 back in
# and re-breaks Unsloth.
!pip install -q wandb pyyaml 2>&1 | tail -5

In [ ]:
import importlib.metadata as md

# If the install silently resolved something odd, see it here rather than
# 40 minutes into a training run.
for pkg in [
    "torch",
    "unsloth",
    "trl",
    "peft",
    "transformers",
    "datasets",
    "bitsandbytes",
    "xformers",
]:
    try:
        print(f"{pkg:<14} {md.version(pkg)}")
    except md.PackageNotFoundError:
        print(f"{pkg:<14} (not installed)")

### Step 2: Verify GPU Hardware
Check that a GPU is active and has sufficient VRAM (T4 = ~15-16GB).

In [ ]:
import torch

assert torch.cuda.is_available(), (
    "GPU is not available! Please enable GPU in Kaggle settings."
)

device_name = torch.cuda.get_device_name(0)
total_vram = torch.cuda.get_device_properties(0).total_memory / (1024**3)
major, minor = torch.cuda.get_device_capability(0)

print(f"Active GPU:  {device_name}  ({torch.cuda.device_count()} visible)")
print(f"VRAM:        {total_vram:.2f} GB")
print(f"Capability:  sm_{major}{minor}")

assert major >= 7, (
    f"{device_name} is sm_{major}{minor}, but sm_70+ is required.\n"
    "Fix: Notebook Settings -> Accelerator -> 'GPU T4 x2', then re-run."
)
print("GPU is compatible with Unsloth + QLoRA.")

### Step 3: Authenticate with HuggingFace & Weights & Biases
Log in to HuggingFace (to load datasets and push model) and Weights & Biases (for live loss tracking).

In [ ]:
import os

from huggingface_hub import login
import wandb

HF_TOKEN = os.environ.get("HF_TOKEN", "")
WANDB_API_KEY = os.environ.get("WANDB_API_KEY", "")

login(token=HF_TOKEN)
print("HuggingFace: authenticated.")

if WANDB_API_KEY:
    try:
        os.environ["WANDB_API_KEY"] = WANDB_API_KEY
        os.environ["WANDB_PROJECT"] = "ios-risk-domain-core"
        wandb.login(key=WANDB_API_KEY)
        REPORT_TO = "wandb"
        print("W&B: authenticated, run will be logged.")
    except Exception as e:
        REPORT_TO = "none"
        print(
            f"W&B: login failed ({type(e).__name__}: {e}). Training continues unlogged."
        )
else:
    REPORT_TO = "none"
    print("W&B: no key set. Training continues unlogged.")

### Step 4: Define Training Configuration & Pipeline

In [ ]:
# Unsloth MUST be imported before transformers / trl / peft.
# It monkey-patches those libraries at import time to install its memory
# optimizations. If they are already in sys.modules, the patches never land —
# you keep the Unsloth API but lose the ~80% memory saving that makes QLoRA
# fit on a 15GB T4. Unsloth warns about this explicitly.
from unsloth import FastLanguageModel

import os
from dataclasses import dataclass, field
from typing import Dict, List, Optional

from datasets import load_dataset


@dataclass
class TrainingConfig:
    # Model & Quantization
    base_model: str = "unsloth/Meta-Llama-3.1-8B-Instruct"
    max_seq_length: int = 1024
    load_in_4bit: bool = True
    resume_from_adapter: Optional[str] = None  # None = fresh LoRA (session 1)

    # LoRA Adapter Config
    lora_r: int = 16
    lora_alpha: int = 16
    lora_dropout: float = 0.0
    target_modules: List[str] = field(
        default_factory=lambda: [
            "q_proj",
            "k_proj",
            "v_proj",
            "o_proj",
            "gate_proj",
            "up_proj",
            "down_proj",
        ]
    )
    bias: str = "none"

    # Dataset
    dataset_name: str = "Etherlabs/ios-risk-finetune-v3"
    max_samples: Optional[int] = None  # v3 is 20,606 rows; train once
    sample_offset: int = 0  # fresh run starts at row zero
    val_size: int = 1000  # fixed loss-monitoring split

    # Optimization
    num_train_epochs: float = 1.0  # TRL defaults to 3 — always set this explicitly
    learning_rate: float = 2e-4
    per_device_train_batch_size: int = 2
    gradient_accumulation_steps: int = 8  # 2 * 8 = 16 effective batch size
    warmup_steps: int = 100
    lr_scheduler_type: str = "cosine"
    weight_decay: float = 0.01
    optim: str = "adamw_8bit"
    seed: int = 42

    # Checkpointing & Logging
    output_dir: str = "/kaggle/working/Llama-3.1-8B-IOS-Risk-v1"
    save_strategy: str = "steps"
    save_steps: int = 250
    eval_steps: int = 250
    save_total_limit: int = 3
    logging_steps: int = 5
    report_to: str = "none"
    wandb_project: str = "ios-risk-domain-core"
    wandb_run_name: Optional[str] = "llama3-8b-ios-risk-v1"


print("Training configuration class ready.")

### Step 5: Dataset Pipeline & Prompt Formatting

In [ ]:
ALPACA_PROMPT = """Below is an instruction that describes a financial risk analysis task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

ALPACA_NO_INPUT_PROMPT = """Below is an instruction that describes a financial risk analysis task. Write a response that appropriately completes the request.

### Instruction:
{}

### Response:
{}"""


def format_instruction_pair(example: Dict[str, str]) -> Dict[str, str]:
    instruction = example.get("instruction", "")
    input_text = example.get("input", "")
    output = example.get("output", "")
    if input_text and input_text.strip():
        text = ALPACA_PROMPT.format(instruction, input_text, output)
    else:
        text = ALPACA_NO_INPUT_PROMPT.format(instruction, output)
    return {"text": text}


def load_and_prepare_dataset(config):
    """
    Loads raw instruction pairs from HuggingFace, formats them into training
    prompts, and returns a (train, eval) pair.

    The final 1,000 shuffled rows monitor loss and are not used for optimization.
    The separate Project 03 test set is held out by source record and CFR section.
    """
    print(f"[Dataset] Loading dataset '{config.dataset_name}' from HuggingFace...")
    dataset = load_dataset(config.dataset_name, split="train")
    total = len(dataset)
    required = {"instruction", "input", "output"}
    assert required.issubset(dataset.column_names), (
        f"Missing columns: {required - set(dataset.column_names)}"
    )
    unique_instructions = len(set(dataset["instruction"]))
    unique_outputs = len(set(dataset["output"]))
    assert unique_instructions >= 3, f"Only {unique_instructions} unique instructions"
    assert unique_outputs >= 100, f"Only {unique_outputs} unique outputs"
    print(
        f"[Dataset] Quality gate: {unique_instructions} instructions, {unique_outputs} outputs"
    )

    # Held-out tail — identical in every session.
    val_start = total - config.val_size
    if val_start <= 0:
        raise ValueError(f"val_size={config.val_size} is >= dataset size {total}")
    eval_raw = dataset.select(range(val_start, total))

    # Training slice walks forward through everything before the val tail.
    start = config.sample_offset
    if start >= val_start:
        raise ValueError(
            f"sample_offset={start} is past the training pool (ends at {val_start}). "
            "The dataset has been fully covered."
        )
    end = (
        val_start
        if config.max_samples is None
        else min(start + config.max_samples, val_start)
    )
    train_raw = dataset.select(range(start, end))

    print(
        f"[Dataset] Train slice: rows [{start:,}, {end:,}) of {val_start:,} available"
    )
    print(
        f"[Dataset] Val set:     rows [{val_start:,}, {total:,}) — fixed across sessions"
    )

    print("[Dataset] Formatting instruction pairs into Alpaca format...")
    train_data = train_raw.map(format_instruction_pair, desc="Formatting train")
    eval_data = eval_raw.map(format_instruction_pair, desc="Formatting val")

    print(
        f"[Dataset] Ready: {len(train_data):,} train samples, {len(eval_data):,} validation samples."
    )
    return train_data, eval_data

### Step 6: Load Model & Attach LoRA Adapters

In [ ]:
def load_model_and_tokenizer(config):
    """
    Loads Llama 3.1 8B in 4-bit and returns it with LoRA adapters attached.

    Two modes:
      * Fresh  — load the base model, then attach new randomly-initialised
                 LoRA adapters. This is session 1.
      * Resume — load a previously trained adapter directory. Unsloth restores
                 the base model AND the trained adapter weights, so training
                 continues from where the last session stopped. Calling
                 get_peft_model() here would wrap a second adapter around the
                 first and silently discard the training you already paid for.
    """
    resuming = bool(config.resume_from_adapter)
    source = config.resume_from_adapter if resuming else config.base_model

    if resuming:
        print(f"[Model] RESUMING from trained adapter '{source}' in 4-bit...")
    else:
        print(
            f"[Model] Loading base model '{source}' in 4-bit (max_seq_length={config.max_seq_length})..."
        )

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=source,
        max_seq_length=config.max_seq_length,
        load_in_4bit=config.load_in_4bit,
    )

    if resuming:
        print("[Model] Trained adapter restored — skipping fresh LoRA attachment.")
        FastLanguageModel.for_training(model)
    else:
        print(
            f"[Model] Attaching LoRA adapters (rank={config.lora_r}, alpha={config.lora_alpha})..."
        )
        model = FastLanguageModel.get_peft_model(
            model,
            r=config.lora_r,
            target_modules=config.target_modules,
            lora_alpha=config.lora_alpha,
            lora_dropout=config.lora_dropout,
            bias=config.bias,
            use_gradient_checkpointing="unsloth",
            random_state=config.seed,
        )

    trainable_params, all_params = model.get_nb_trainable_parameters()
    trainable_pct = 100 * trainable_params / all_params
    print(
        f"[Model] Parameter Summary: {trainable_params:,} trainable / {all_params:,} total ({trainable_pct:.3f}% trainable)"
    )

    return model, tokenizer

### Step 7: Launch Training Run
Execute the training loop and monitor loss convergence live.

In [ ]:
import dataclasses
import inspect

import torch
from trl import SFTConfig, SFTTrainer

# This is a fresh v3 run. Do not attach v1 or v2: both used datasets that later
# failed quality or evaluation-leakage checks.
config = TrainingConfig(
    max_samples=None,
    sample_offset=0,
    resume_from_adapter=None,
    output_dir="/kaggle/working/Llama-3.1-8B-IOS-Risk-v1",
    wandb_run_name="llama3-8b-ios-risk-v1",
    report_to=REPORT_TO,
)
print("[Preflight] fresh v3 run; no previous adapter will be loaded")

train_data, eval_data = load_and_prepare_dataset(config)
model, tokenizer = load_model_and_tokenizer(config)
assert tokenizer.eos_token and tokenizer.eos_token_id is not None, (
    "Tokenizer has no EOS token"
)


def append_eos(example):
    text = example["text"]
    return {
        "text": text
        if text.endswith(tokenizer.eos_token)
        else text + tokenizer.eos_token
    }


train_data = train_data.map(append_eos, desc="Appending EOS to train")
eval_data = eval_data.map(append_eos, desc="Appending EOS to val")

fields = {field.name for field in dataclasses.fields(SFTConfig)}
sft_kwargs = dict(
    output_dir=config.output_dir,
    per_device_train_batch_size=config.per_device_train_batch_size,
    gradient_accumulation_steps=config.gradient_accumulation_steps,
    num_train_epochs=config.num_train_epochs,
    learning_rate=config.learning_rate,
    lr_scheduler_type=config.lr_scheduler_type,
    warmup_steps=config.warmup_steps,
    weight_decay=config.weight_decay,
    optim=config.optim,
    logging_steps=config.logging_steps,
    eval_steps=config.eval_steps,
    save_strategy=config.save_strategy,
    save_steps=config.save_steps,
    save_total_limit=config.save_total_limit,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    report_to=config.report_to,
    run_name=config.wandb_run_name,
    seed=config.seed,
    dataset_text_field="text",
    dataset_num_proc=2,
    packing=False,
)
sft_kwargs["eval_strategy" if "eval_strategy" in fields else "evaluation_strategy"] = (
    "steps"
)
sft_kwargs["max_length" if "max_length" in fields else "max_seq_length"] = (
    config.max_seq_length
)
for key in [key for key in sft_kwargs if key not in fields]:
    sft_kwargs.pop(key)

training_args = SFTConfig(**sft_kwargs)
params = inspect.signature(SFTTrainer.__init__).parameters
tokenizer_key = "processing_class" if "processing_class" in params else "tokenizer"
trainer = SFTTrainer(
    model=model,
    train_dataset=train_data,
    eval_dataset=eval_data,
    args=training_args,
    **{tokenizer_key: tokenizer},
)

# This assertion checks the tokenized data that the trainer will actually use.
# If EOS is missing, the notebook stops here before spending training hours.
sample_ids = trainer.train_dataset[0].get("input_ids")
assert sample_ids, "Trainer did not expose tokenized input_ids"
assert sample_ids[-1] == tokenizer.eos_token_id, (
    f"EOS preflight failed: got {sample_ids[-1]}, expected {tokenizer.eos_token_id}"
)
print(f"[Preflight] EOS verified on actual trainer input: {tokenizer.eos_token_id}")

print("Starting training run on GPU...")
train_result = trainer.train()

os.makedirs(config.output_dir, exist_ok=True)
model.save_pretrained(config.output_dir)
tokenizer.save_pretrained(config.output_dir)
print(f"LoRA adapter successfully saved to {config.output_dir}")

### Step 8: Live Domain Verification (Inference Test)
Prompt the fine-tuned model with a domain-specific financial risk scenario to verify its outputs.

In [ ]:
FastLanguageModel.for_inference(model)

ALPACA = """Below is an instruction that describes a financial risk analysis task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
"""

# Same shape as eval/testset.py cases. Expected: HIGH RISK / CARD TESTING.
RISK_INSTRUCTION = (
    "You are IOS Risk, an AI system for financial risk assessment. "
    "Analyse the following transaction and assess its fraud risk."
)
CLASSIFY_INSTRUCTION = "Classify this financial transaction as FRAUD or LEGITIMATE based on the features provided."

checks = [
    (
        RISK_INSTRUCTION,
        "Amount: $1.12 | Hour: 3 | TxnCount1h: 22 | MicroTxn: 1 | OffHours: 1 | LargeTxn: 0 | RoundAmt: 0",
        "expect HIGH RISK - card testing",
        200,
    ),
    (
        RISK_INSTRUCTION,
        "Amount: $38.40 | Hour: 14 | TxnCount1h: 2 | MicroTxn: 0 | OffHours: 0 | LargeTxn: 0 | RoundAmt: 0",
        "expect LOW RISK - legitimate",
        200,
    ),
    (
        CLASSIFY_INSTRUCTION,
        "Amount: $9900.00 | Hour: 11 | OffHours: 0 | MicroTxn: 0 | RoundAmt: 1 | LargeTxn: 1 | AmtZscore: 4.210 | TxnCount1h: 4",
        "expect FRAUD or LEGITIMATE, one word",
        8,
    ),
]

for instruction, features, note, max_new in checks:
    inputs = tokenizer([ALPACA.format(instruction, features)], return_tensors="pt").to(
        "cuda"
    )
    out = model.generate(
        **inputs,
        max_new_tokens=max_new,
        use_cache=True,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )
    reply = tokenizer.decode(
        out[0][inputs["input_ids"].shape[1] :], skip_special_tokens=True
    )
    print(f"--- {note} ---")
    print(features)
    print("->", reply.strip()[:400])
    print()

### Step 9: Push Adapter to HuggingFace Hub (Optional / Final Run)
When ready, publish the adapter to the Etherlabs organization on HuggingFace.

In [ ]:
# Publish the adapter — ONLY after eval/domain_eval.py clears the Project 03
# targets (tier_accuracy > 0.70, avg_quality > 0.60). Publishing an adapter
# that fails its own eval is what produced the v1 situation.
#
# model.push_to_hub("Etherlabs/Llama-3.1-8B-IOS-Risk-v1", token=HF_TOKEN)
# tokenizer.push_to_hub("Etherlabs/Llama-3.1-8B-IOS-Risk-v1", token=HF_TOKEN)
